# MDI3003 — Advanced Predictive Analytics
## Experiment 06: Time-Series Analysis and Forecasting of Reported Crime Incidents by Time and Location using AR and ARIMA Models

**Dataset used:** D2 — NYPD Complaint Data (Current, Year-To-Date) — `NYPD_Complaint_Data_Current__Year_To_Date__20260901.csv`
**Location field used:** `BORO_NM` (borough) — one borough is used as the core series; a second borough is used for the required replication step.
**Frequency:** Weekly (`W-MON`)

| Field | Value |
|---|---|
| **Student Name** | Harshita |
| **Registration Number** | 23MID0043 |
| **Course Code** | MDI3003 |
| **Faculty** | Dr. Durgesh Kumar, Assistant Professor (Senior), SCOPE, VIT Vellore |
| **Semester** | Fall Semester 2026-2027 |


In [ ]:
# ============================================================
# STUDENT / SUBMISSION IDENTIFICATION — EDIT THESE BEFORE SUBMITTING
# ============================================================
STUDENT_NAME = "Harshita"                 # <-- edit
REGISTRATION_NUMBER = "23MID0043"          # <-- edit
COURSE_CODE = "MDI3003"
LAB_TITLE = "Lab06_Crime_AR_ARIMA"
FACULTY = "Dr. Durgesh Kumar, Assistant Professor (Senior), SCOPE, VIT Vellore"
SEMESTER = "Fall Semester 2026-2027"

print(f"{REGISTRATION_NUMBER}_{LAB_TITLE}")


---
## 0. Colab Setup

This cell installs/imports everything needed and mounts nothing — the dataset is uploaded directly into the Colab
session's file system (see the upload cell in Section 1). All outputs (figures, CSVs, JSON manifest, and the notebook
itself) are written to a local `lab06_outputs/` folder and, in the final cell, **automatically zipped and downloaded**
to your machine — no Google Drive mount required.


In [ ]:
# ============================================================
# 0. ENVIRONMENT SETUP
# ============================================================
!pip -q install pandas numpy matplotlib statsmodels scikit-learn joblib

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json, platform, sys, time, zipfile, shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

SEED = 42
np.random.seed(SEED)

OUT = Path('lab06_outputs')
FIG = OUT / 'figures'
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True, parents=True)

plt.rcParams['figure.dpi'] = 100
print("Environment ready.")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

---
## 1. Dataset Provenance and Governance (Manual §7, §8)

| Field | Record |
|---|---|
| Dataset | D2 — NYPD Complaint Data (Current, Year-To-Date) |
| Agency | NYC Open Data / New York City Police Department (NYPD) |
| Official source URL | https://data.cityofnewyork.us/Public-Safety/NYPD-Complaint-Data-Current-Year-To-Date-/5uac-w243 |
| File used | `NYPD_Complaint_Data_Current__Year_To_Date__20260901.csv` |
| Access / extract date | 2026-09-01 (filename-stamped) |
| Date field used | `CMPLNT_FR_DT` (complaint **occurrence** "from" date) — chosen over `RPT_DT` (report date) to model when incidents actually occurred rather than administrative reporting lag |
| Location field used | `BORO_NM` (borough — NYC's coarsest, most reliable geographic unit; precinct `ADDR_PCT_CD` is available for finer-grained advanced work) |
| Identifier | `CMPLNT_NUM` — used only for deduplication, never as a predictive feature |
| Known caveats | `(null)` values appear in `BORO_NM` and other fields; a small number of malformed/erroneous `CMPLNT_FR_DT` values exist (e.g. non-current-year dates) and must be audited before aggregation; NYPD data reflects **reported/recorded** complaints, not actual crime prevalence |

Upload the CSV below (or, if running outside Colab with the file already in the working directory, the upload step is skipped automatically).


In [ ]:
# ============================================================
# 1.1 LOAD DATA (Colab upload widget, with local-file fallback)
# ============================================================
DATA_FILENAME = "NYPD_Complaint_Data_Current__Year_To_Date__20260901.csv"
if not Path(DATA_FILENAME).exists() and (Path("datasets") / DATA_FILENAME).exists():
    DATA_FILENAME = str(Path("datasets") / DATA_FILENAME)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not Path(DATA_FILENAME).exists():
    from google.colab import files
    print(f"Please upload: {DATA_FILENAME}")
    uploaded = files.upload()
    if not Path(DATA_FILENAME).exists():
        csv_uploaded = [f for f in uploaded.keys() if f.lower().endswith(".csv")]
        if csv_uploaded:
            DATA_FILENAME = csv_uploaded[0]

assert Path(DATA_FILENAME).exists(), f"Dataset not found: {DATA_FILENAME}. Upload it or place it in the working directory or datasets/."
print("Using data file:", DATA_FILENAME)


In [ ]:
# ============================================================
# 1.2 CONFIGURATION (Manual §12.1)
# ============================================================
CONFIG = {
    'dataset': 'NYPD Complaint Data - Current (Year To Date)',
    'dataset_url': 'https://data.cityofnewyork.us/Public-Safety/NYPD-Complaint-Data-Current-Year-To-Date-/5uac-w243',
    'access_date': '2026-09-01',
    'date_col': 'CMPLNT_FR_DT',
    'location_col': 'BORO_NM',
    'location_value': 'BROOKLYN',        # core-experiment location (highest-volume borough)
    'second_location_value': 'MANHATTAN', # required replication location (Manual step 18.1)
    'category_col': 'OFNS_DESC',
    'category_value': None,               # set e.g. 'GRAND LARCENY' to filter to one offense type
    'frequency': 'W-MON',                 # weekly series, weeks starting Monday
    'test_periods': 12,                   # locked future holdout (weeks)
    'valid_year_min': 2026,               # sanity bound: this extract is "current year to date"
    'seed': SEED,
}
print(json.dumps(CONFIG, indent=2))

---
## 2. Load, Parse and Audit (Manual §12.2)

We parse timestamps, check for duplicate `CMPLNT_NUM` values, and flag rows with implausible dates
(the raw extract contains a small number of corrupted years, e.g. `1016`) before building any time series.


In [ ]:
# ============================================================
# 2. LOAD AND AUDIT
# ============================================================
usecols = [CONFIG['date_col'], CONFIG['location_col'], 'CMPLNT_NUM',
           CONFIG['category_col'], 'LAW_CAT_CD', 'ADDR_PCT_CD']

df_raw = pd.read_csv(DATA_FILENAME, usecols=usecols, low_memory=False)
print("Raw shape:", df_raw.shape)

# Required-column check
required = {CONFIG['date_col'], CONFIG['location_col']}
missing = required - set(df_raw.columns)
assert not missing, f'Missing required columns: {missing}'

# Duplicate audit (identifiers used for dedup only, never as features)
dupe_count = df_raw['CMPLNT_NUM'].duplicated().sum()
print(f"Duplicate CMPLNT_NUM rows: {dupe_count}")

# Parse dates
df = df_raw.copy()
df[CONFIG['date_col']] = pd.to_datetime(df[CONFIG['date_col']], format='%m/%d/%Y', errors='coerce')

n_before = len(df)
df = df.dropna(subset=[CONFIG['date_col']]).copy()
n_after_parse = len(df)
print(f"Dropped {n_before - n_after_parse} rows with unparseable dates.")

# Audit implausible years (data-quality issue documented per Manual governance checklist)
bad_year_mask = (df[CONFIG['date_col']].dt.year < CONFIG['valid_year_min'] - 1) | \
                (df[CONFIG['date_col']].dt.year > pd.Timestamp.today().year)
print(f"Rows with implausible occurrence year (<{CONFIG['valid_year_min']-1} or in the future): {bad_year_mask.sum()}")
df = df.loc[~bad_year_mask].copy()

# Drop rows with null/missing location
df = df[~df[CONFIG['location_col']].isin(['(null)', None]) & df[CONFIG['location_col']].notna()].copy()

print("Clean shape after audit:", df.shape)
print(df[[CONFIG['date_col'], CONFIG['location_col']]].head())
print("\nDate range:", df[CONFIG['date_col']].min(), "to", df[CONFIG['date_col']].max())
print("\nBorough counts:\n", df[CONFIG['location_col']].value_counts())

---
## 3. Construct a Regular Location-Level Weekly Series (Manual §12.3)

Location defines *which* series is modeled — it is never used as a numeric ARIMA feature.


In [ ]:
# ============================================================
# 3. BUILD THE CORE LOCATION SERIES
# ============================================================
def build_series(data, location_value, config):
    loc = data[data[config['location_col']].astype(str) == str(location_value)].copy()
    if config.get('category_value'):
        loc = loc[loc[config['category_col']].astype(str).str.upper() == config['category_value'].upper()]
    assert len(loc) > 0, f'Selected location has no observations: {location_value}'

    y = (loc.set_index(config['date_col'])
           .resample(config['frequency'])
           .size()
           .rename('incidents')
           .asfreq(config['frequency'], fill_value=0))

    assert y.index.is_monotonic_increasing
    assert y.index.is_unique
    assert y.isna().sum() == 0
    return y, loc

y, loc_df = build_series(df, CONFIG['location_value'], CONFIG)
print(f"Series for {CONFIG['location_value']}: {len(y)} weekly periods")
print(y.describe())

In [ ]:
# ============================================================
# 3.1 VISUALIZE THE RAW SERIES
# ============================================================
fig, ax = plt.subplots(figsize=(11,4))
y.plot(ax=ax, title=f"Weekly reported incident counts — {CONFIG['location_value']}")
ax.set_ylabel('Incidents per week')
ax.set_xlabel('Week')
plt.tight_layout()
plt.savefig(FIG / '01_raw_series.png', bbox_inches='tight')
plt.show()
print("Interpretation: the plot shows the weekly volume of reported incidents for the selected borough. "
      "Any visible trend or level shift should be cross-checked against known reporting/policy changes rather "
      "than assumed to reflect actual crime prevalence changes.")

In [ ]:
# ============================================================
# 3.2 EXPLORATORY DECOMPOSITION: rolling mean/variance, weekday/monthly patterns
# ============================================================
roll = y.rolling(4).mean()
roll_std = y.rolling(4).std()

fig, ax = plt.subplots(figsize=(11,4))
y.plot(ax=ax, alpha=0.4, label='Weekly incidents')
roll.plot(ax=ax, label='4-week rolling mean', linewidth=2)
roll_std.plot(ax=ax, label='4-week rolling std', linewidth=1, linestyle='--')
ax.set_title(f"Rolling mean/variance — {CONFIG['location_value']}")
ax.set_ylabel('Incidents per week')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / '02_rolling_stats.png', bbox_inches='tight')
plt.show()
print("Interpretation: a roughly stable rolling mean/std over time is consistent with (weak) stationarity; "
      "systematic drift in either suggests trend or heteroscedasticity that later diagnostics should confirm.")

---
## 4. Chronological Train/Test Split and Naive Baseline (Manual §12.4)

No shuffling. Training data = all periods except the final locked test horizon.


In [ ]:
# ============================================================
# 4. CHRONOLOGICAL SPLIT + NAIVE BASELINE
# ============================================================
H = CONFIG['test_periods']
assert len(y) > 3*H, 'Series is too short for the chosen holdout.'

train, test = y.iloc[:-H], y.iloc[-H:]
print(f"Train periods: {len(train)}  |  Test periods: {len(test)}")
print("Train range:", train.index.min(), "->", train.index.max())
print("Test range :", test.index.min(), "->", test.index.max())

naive_pred = np.repeat(train.iloc[-1], len(test))

def score(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5
    }

naive_scores = score(test, naive_pred)
print('Naive (last-value persistence):', naive_scores)

---
## 5. Stationarity and Lag Diagnostics (Manual §12.5)

ADF test and ACF/PACF are computed on **training data only** — the test period is never consulted before
model-order selection.


In [ ]:
# ============================================================
# 5. ADF TEST
# ============================================================
adf_stat, adf_p, adf_lags, adf_nobs, adf_crit, _ = adfuller(train)
print(f"ADF statistic = {adf_stat:.4f}")
print(f"p-value       = {adf_p:.4f}")
print(f"lags used     = {adf_lags}")
print("critical values:", adf_crit)

if adf_p < 0.05:
    print("\n=> Reject unit-root null at 5%: training series looks stationary (or close to it).")
else:
    print("\n=> Fail to reject unit-root null at 5%: differencing (d>=1) is likely warranted.")

In [ ]:
# ============================================================
# 5.1 ACF / PACF ON TRAINING DATA
# ============================================================
fig, ax = plt.subplots(1, 2, figsize=(12,3.8))
plot_acf(train, lags=min(30, len(train)//4), ax=ax[0])
ax[0].set_title('ACF (training only)')
plot_pacf(train, lags=min(30, len(train)//4), ax=ax[1], method='ywm')
ax[1].set_title('PACF (training only)')
plt.tight_layout()
plt.savefig(FIG / '03_acf_pacf.png', bbox_inches='tight')
plt.show()
print("Interpretation: slowly-decaying ACF suggests non-stationarity/trend (motivating differencing); "
      "a PACF that cuts off sharply after lag p motivates an AR(p) order.")

---
## 6. Autoregressive Model AR(p) (Manual §12.6, §13)


In [ ]:
# ============================================================
# 6. FIT AR MODEL
# ============================================================
AR_LAGS = 4   # justified from training PACF above; adjust if PACF suggests otherwise
ar_model = AutoReg(train, lags=AR_LAGS, old_names=False, trend='ct').fit()
ar_pred = ar_model.predict(start=len(train), end=len(train)+len(test)-1, dynamic=False)
ar_pred.index = test.index

ar_scores = score(test, ar_pred)
print(ar_model.summary())
print('\nAR test-set scores:', ar_scores)

---
## 7. ARIMA(p,d,q) — Training-Selected Candidate Search (Manual §12.7, §13)

Order is chosen using **training-only AIC**; the test set is evaluated exactly once, afterward.


In [ ]:
# ============================================================
# 7. ARIMA CANDIDATE SEARCH (small, training-only)
# ============================================================
candidates = [(1,0,0), (2,0,0), (1,1,1), (2,1,1), (2,1,2), (3,1,1)]
rows = []
for order in candidates:
    try:
        m = ARIMA(train, order=order).fit()
        rows.append({'order': order, 'AIC': m.aic, 'BIC': m.bic, 'model': m})
    except Exception as e:
        print('Failed', order, e)

candidate_table = pd.DataFrame([{'order': r['order'], 'AIC': r['AIC'], 'BIC': r['BIC']} for r in rows]) \
                    .sort_values('AIC').reset_index(drop=True)
print(candidate_table)

ranked = sorted(rows, key=lambda z: z['AIC'])
best = ranked[0]
print('\nTraining-selected order:', best['order'], ' AIC =', round(best['AIC'],2))
arima_model = best['model']
arima_fc_obj = arima_model.get_forecast(steps=len(test))
arima_pred = arima_fc_obj.predicted_mean
arima_pred.index = test.index
arima_ci = arima_fc_obj.conf_int(alpha=0.05)
arima_ci.index = test.index

arima_scores = score(test, arima_pred)
print('ARIMA test-set scores:', arima_scores)

---
## 8. Forecast Comparison, Prediction Intervals, and Residual Diagnostics (Manual §12.8, §14)


In [ ]:
# ============================================================
# 8.1 MODEL COMPARISON TABLE
# ============================================================
results = pd.DataFrame([
    {'Model': 'Naive', **naive_scores},
    {'Model': f'AR({AR_LAGS})', **ar_scores},
    {'Model': f'ARIMA{best["order"]}', **arima_scores},
]).sort_values('MAE').reset_index(drop=True)

print(results)
results.to_csv(OUT / 'model_comparison.csv', index=False)

In [ ]:
# ============================================================
# 8.2 FORECAST PLOT (actual vs naive/AR/ARIMA) WITH PREDICTION INTERVAL
# ============================================================
pred_df = pd.DataFrame({
    'actual': test,
    'naive': naive_pred,
    'AR': np.asarray(ar_pred),
    'ARIMA': np.asarray(arima_pred),
}, index=test.index)

fig, ax = plt.subplots(figsize=(11,4.5))
train.iloc[-20:].plot(ax=ax, label='Train (last 20 wks)', color='gray')
pred_df['actual'].plot(ax=ax, marker='o', label='Actual', color='black')
pred_df['naive'].plot(ax=ax, marker='.', linestyle='--', label='Naive')
pred_df['AR'].plot(ax=ax, marker='.', linestyle='--', label=f'AR({AR_LAGS})')
pred_df['ARIMA'].plot(ax=ax, marker='.', linestyle='--', label=f'ARIMA{best["order"]}')
ax.fill_between(test.index, arima_ci.iloc[:,0], arima_ci.iloc[:,1], color='orange', alpha=0.15,
                 label='ARIMA 95% CI')
ax.set_title(f"Locked future holdout: actual vs forecast — {CONFIG['location_value']}")
ax.set_ylabel('Reported incidents')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / '04_forecast_comparison.png', bbox_inches='tight')
plt.show()

# empirical coverage of the ARIMA 95% interval
covered = ((test.values >= arima_ci.iloc[:,0].values) & (test.values <= arima_ci.iloc[:,1].values)).mean()
print(f"Empirical coverage of ARIMA's nominal 95% interval on the test set: {covered:.0%} "
      "(interpret cautiously with only "+str(len(test))+" test points; not a guarantee of future coverage).")

pred_df.to_csv(OUT / 'test_predictions.csv')

In [ ]:
# ============================================================
# 8.3 NEGATIVE-FORECAST / COUNT-DATA CAVEAT CHECK (Manual §14.3)
# ============================================================
neg_ar = (ar_pred < 0).sum()
neg_arima = (arima_pred < 0).sum()
print(f"AR forecast periods with negative predicted counts: {neg_ar}")
print(f"ARIMA forecast periods with negative predicted counts: {neg_arima}")
print("Per the manual, negative forecasts are reported as a Gaussian-ARIMA-on-count-data limitation "
      "and are NOT silently clipped here; see Section 15 (advanced) for a log1p-transformed comparison.")

In [ ]:
# ============================================================
# 8.4 RESIDUAL DIAGNOSTICS: plot, ACF, Ljung-Box
# ============================================================
resid = arima_model.resid.dropna()

fig, ax = plt.subplots(1, 2, figsize=(12,3.8))
ax[0].plot(resid.index, resid.values)
ax[0].axhline(0, color='red', linestyle='--', linewidth=1)
ax[0].set_title(f'ARIMA{best["order"]} residuals over time')
plot_acf(resid, lags=min(24, len(resid)//3), ax=ax[1])
ax[1].set_title('Residual ACF')
plt.tight_layout()
plt.savefig(FIG / '05_residual_diagnostics.png', bbox_inches='tight')
plt.show()

lb_lags = min(10, max(2, len(resid)//10))
lb = acorr_ljungbox(resid, lags=[lb_lags], return_df=True)
print(lb)
if lb['lb_pvalue'].iloc[0] < 0.05:
    print(f"\n=> Ljung-Box p<0.05 at lag {lb_lags}: residuals show remaining autocorrelation — model is not fully adequate.")
else:
    print(f"\n=> Ljung-Box p>=0.05 at lag {lb_lags}: no strong evidence of remaining residual autocorrelation.")

---
## 9. Rolling-Origin / Walk-Forward Validation on Training History (Manual §6, §18.1, Appendix B.1)

Required in the *core* (not just advanced): repeated forecasts from successive historical cutoffs,
using **training/validation history only** (the final locked test set from Section 4 is untouched here).


In [ ]:
# ============================================================
# 9. ROLLING-ORIGIN BACKTEST (on train, comparing AR vs ARIMA stability)
# ============================================================
def rolling_origins(series, initial, horizon, step):
    origins = []
    end = initial
    while end + horizon <= len(series):
        origins.append((series.iloc[:end], series.iloc[end:end+horizon]))
        end += step
    return origins

folds = rolling_origins(train, initial=max(30, len(train)//2), horizon=8, step=8)
print(f"Number of rolling-origin folds on training history: {len(folds)}")

fold_rows = []
for i, (tr, te) in enumerate(folds, 1):
    try:
        m_ar = AutoReg(tr, lags=AR_LAGS, old_names=False, trend='ct').fit()
        p_ar = m_ar.predict(start=len(tr), end=len(tr)+len(te)-1, dynamic=False)
        s_ar = score(te, p_ar)
    except Exception as e:
        s_ar = {'MAE': np.nan, 'RMSE': np.nan}

    try:
        m_ai = ARIMA(tr, order=best['order']).fit()
        p_ai = m_ai.forecast(len(te))
        s_ai = score(te, p_ai)
    except Exception as e:
        s_ai = {'MAE': np.nan, 'RMSE': np.nan}

    fold_rows.append({'fold': i, 'AR_MAE': s_ar['MAE'], 'AR_RMSE': s_ar['RMSE'],
                       'ARIMA_MAE': s_ai['MAE'], 'ARIMA_RMSE': s_ai['RMSE']})

fold_df = pd.DataFrame(fold_rows)
print(fold_df)
print("\nMean +/- SD across folds:")
summary = fold_df[['AR_MAE','ARIMA_MAE','AR_RMSE','ARIMA_RMSE']].agg(['mean','std'])
print(summary)
fold_df.to_csv(OUT / 'rolling_origin_results.csv', index=False)

fig, ax = plt.subplots(figsize=(9,4))
ax.plot(fold_df['fold'], fold_df['AR_MAE'], marker='o', label='AR MAE')
ax.plot(fold_df['fold'], fold_df['ARIMA_MAE'], marker='o', label='ARIMA MAE')
ax.set_xlabel('Fold'); ax.set_ylabel('MAE'); ax.set_title('Rolling-origin MAE by fold')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / '06_rolling_origin.png', bbox_inches='tight')
plt.show()
print("Interpretation: lower mean AND lower variance in fold-wise error indicates the more stable model "
      "across different historical cutoffs, not just a single lucky split.")

---
## 10. Second-Location Replication — Time-and-Location Analysis (Manual §15, §18.1)

The exact same protocol (frequency, split logic, naive/AR/ARIMA, evaluation) is replicated on a second
borough, holding the time window and horizon fixed for a fair comparison.


In [ ]:
# ============================================================
# 10. REPLICATE ON SECOND LOCATION
# ============================================================
y2, loc2_df = build_series(df, CONFIG['second_location_value'], CONFIG)
train2, test2 = y2.iloc[:-H], y2.iloc[-H:]

naive_pred2 = np.repeat(train2.iloc[-1], len(test2))
ar_model2 = AutoReg(train2, lags=AR_LAGS, old_names=False, trend='ct').fit()
ar_pred2 = ar_model2.predict(start=len(train2), end=len(train2)+len(test2)-1, dynamic=False)

m2 = ARIMA(train2, order=best['order']).fit()   # same training-selected order, for a fair comparison
arima_pred2 = m2.forecast(len(test2))

results2 = pd.DataFrame([
    {'Location': CONFIG['second_location_value'], 'Model': 'Naive', **score(test2, naive_pred2)},
    {'Location': CONFIG['second_location_value'], 'Model': f'AR({AR_LAGS})', **score(test2, ar_pred2)},
    {'Location': CONFIG['second_location_value'], 'Model': f'ARIMA{best["order"]}', **score(test2, arima_pred2)},
])
results1_labeled = results.copy(); results1_labeled.insert(0, 'Location', CONFIG['location_value'])
two_location_comparison = pd.concat([results1_labeled, results2], ignore_index=True)
print(two_location_comparison)
two_location_comparison.to_csv(OUT / 'two_location_comparison.csv', index=False)

fig, ax = plt.subplots(1, 2, figsize=(13,4), sharey=False)
y.plot(ax=ax[0], title=CONFIG['location_value'])
y2.plot(ax=ax[1], title=CONFIG['second_location_value'], color='darkorange')
for a in ax: a.set_ylabel('Incidents/week')
plt.tight_layout()
plt.savefig(FIG / '07_two_location_series.png', bbox_inches='tight')
plt.show()
print(f"Mean weekly incidents — {CONFIG['location_value']}: {y.mean():.1f} (std {y.std():.1f}) | "
      f"{CONFIG['second_location_value']}: {y2.mean():.1f} (std {y2.std():.1f})")
print("Interpretation: differences in level/volatility/model error across boroughs should be attributed to "
      "differences in reported-incident volume and dynamics, not compared as if boroughs share identical "
      "reporting/enforcement processes.")

---
## 11. ADVANCED EXTENSION A — SARIMA for Seasonality (Manual §16.1)

This year-to-date extract does not span multiple full annual cycles, so a 52-week seasonal period cannot be
reliably estimated. We instead test a defensible **monthly-scale seasonal period (≈4 weeks)** as a diagnostic
exercise, and explicitly flag the annual-seasonality limitation.


In [ ]:
# ============================================================
# 11. SARIMA (seasonal period chosen defensibly given data span)
# ============================================================
n_weeks_available = len(y)
print(f"Weeks of history available: {n_weeks_available}")

# An annual (52-week) seasonal period would need several years of history to estimate reliably;
# this YTD extract does not have that. We use s=4 (approx. monthly cycle) as an illustrative,
# data-supported alternative and document the limitation explicitly.
SEASONAL_PERIOD = 4

sarima = SARIMAX(train, order=best['order'], seasonal_order=(1,0,1,SEASONAL_PERIOD),
                  enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
sarima_pred = sarima.forecast(len(test))
sarima_scores = score(test, sarima_pred)
print('SARIMA scores:', sarima_scores)
print(sarima.summary().tables[0])

print("\nLIMITATION: with only", n_weeks_available, "weeks of history, a true annual (s=52) seasonal "
      "component cannot be reliably fit (needs multiple full yearly cycles). The s=4 result above is an "
      "illustrative diagnostic, not a validated annual-seasonality model. Re-run with a multi-year historic "
      "extract (D2's companion 'NYPD Complaint Data Historic' dataset) for a genuine annual SARIMA.")

---
## 12. ADVANCED EXTENSION B — Count-Data-Aware Comparison: log1p-ARIMA (Manual §14.3, §16.5)

Since ARIMA assumes a Gaussian, real-valued process while incident counts are non-negative, we compare the
plain ARIMA above against a `log1p`-transformed ARIMA that structurally cannot forecast negative counts.


In [ ]:
# ============================================================
# 12. LOG1P-TRANSFORMED ARIMA vs PLAIN ARIMA
# ============================================================
train_log = np.log1p(train)
m_log = ARIMA(train_log, order=best['order']).fit()
pred_log = m_log.forecast(len(test))
pred_log_back = np.expm1(pred_log)   # invert transform back to count scale

log_scores = score(test, pred_log_back)

comparison_countaware = pd.DataFrame([
    {'Model': f'ARIMA{best["order"]} (raw)', **arima_scores,
     'Negative forecasts': int((arima_pred < 0).sum())},
    {'Model': f'log1p-ARIMA{best["order"]}', **log_scores,
     'Negative forecasts': int((pred_log_back < 0).sum())},
])
print(comparison_countaware)
comparison_countaware.to_csv(OUT / 'count_aware_comparison.csv', index=False)

fig, ax = plt.subplots(figsize=(10,4))
test.plot(ax=ax, marker='o', label='Actual', color='black')
pd.Series(np.asarray(arima_pred), index=test.index).plot(ax=ax, marker='.', linestyle='--', label='ARIMA (raw)')
pd.Series(np.asarray(pred_log_back), index=test.index).plot(ax=ax, marker='.', linestyle='--', label='log1p-ARIMA')
ax.set_title('Count-aware comparison: raw vs log1p-transformed ARIMA')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / '08_count_aware_comparison.png', bbox_inches='tight')
plt.show()
print("Discussion: log1p-ARIMA guarantees non-negative back-transformed forecasts and is a lightweight "
      "count-aware alternative; a full treatment would use Poisson/negative-binomial autoregressive or "
      "state-space count models (Manual §16.5), left as further work.")

---
## 13. ADVANCED EXTENSION C — SARIMAX with Justified Calendar Exogenous Features (Manual §16.3)

Calendar indicators (`is_summer`, `week_of_year` harmonics) are known at forecast time by construction and
are used as exogenous regressors — no future-realized information is used.


In [ ]:
# ============================================================
# 13. SARIMAX WITH CALENDAR EXOGENOUS FEATURES
# ============================================================
def calendar_exog(index):
    woy = index.isocalendar().week.astype(float)
    return pd.DataFrame({
        'sin52': np.sin(2*np.pi*woy/52),
        'cos52': np.cos(2*np.pi*woy/52),
    }, index=index)

exog_train = calendar_exog(train.index)
exog_test = calendar_exog(test.index)

sarimax_model = SARIMAX(train, order=best['order'], exog=exog_train,
                         enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
sarimax_pred = sarimax_model.forecast(len(test), exog=exog_test)
sarimax_scores = score(test, sarimax_pred)
print('SARIMAX (calendar harmonics) scores:', sarimax_scores)
print('Plain ARIMA scores for comparison:', arima_scores)

print("\nAll exogenous features here (sin/cos of week-of-year) are deterministic calendar functions, "
      "known at any future forecast origin by construction — they cannot leak future information.")

---
## 14. ADVANCED EXTENSION D — Many-Location Replication (Manual §15, §18.2)

The identical protocol is replicated across **all five NYC boroughs** and summarized, without any
person-level or address-level inference.


In [ ]:
# ============================================================
# 14. FIVE-BOROUGH REPLICATION
# ============================================================
all_boroughs = sorted(df[CONFIG['location_col']].dropna().unique().tolist())
print("Boroughs found:", all_boroughs)

multi_rows = []
borough_series = {}
for b in all_boroughs:
    try:
        yb, _ = build_series(df, b, CONFIG)
        borough_series[b] = yb
        trb, teb = yb.iloc[:-H], yb.iloc[-H:]
        nv = np.repeat(trb.iloc[-1], len(teb))
        arb = AutoReg(trb, lags=AR_LAGS, old_names=False, trend='ct').fit()
        arb_pred = arb.predict(start=len(trb), end=len(trb)+len(teb)-1, dynamic=False)
        mib = ARIMA(trb, order=best['order']).fit()
        mib_pred = mib.forecast(len(teb))

        multi_rows.append({
            'Borough': b, 'n_periods': len(yb), 'mean_weekly': yb.mean(), 'std_weekly': yb.std(),
            'Naive_MAE': score(teb, nv)['MAE'],
            'AR_MAE': score(teb, arb_pred)['MAE'],
            'ARIMA_MAE': score(teb, mib_pred)['MAE'],
        })
    except Exception as e:
        print(f"Skipped {b}: {e}")

multi_df = pd.DataFrame(multi_rows).sort_values('mean_weekly', ascending=False)
print(multi_df)
multi_df.to_csv(OUT / 'multi_borough_replication.csv', index=False)

fig, ax = plt.subplots(figsize=(10,4))
for b, s in borough_series.items():
    (s / s.mean()).plot(ax=ax, label=b, alpha=0.8)  # normalized for comparability of shape
ax.set_title('Normalized weekly incident series by borough (each series / its own mean)')
ax.set_ylabel('Relative level')
ax.legend()
plt.tight_layout()
plt.savefig(FIG / '09_multi_borough_series.png', bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(9,4))
multi_df.set_index('Borough')[['Naive_MAE','AR_MAE','ARIMA_MAE']].plot(kind='bar', ax=ax)
ax.set_title('Test-set MAE by borough and model')
ax.set_ylabel('MAE (incidents/week)')
plt.tight_layout()
plt.savefig(FIG / '10_multi_borough_mae.png', bbox_inches='tight')
plt.show()
print("Interpretation: this is an area-level comparison of reported-incident forecastability only; it must "
      "not be read as a person-level or neighborhood-danger ranking (Manual §23).")

---
## 15. ADVANCED EXTENSION E — Category-Specific Series (Manual §18.2)

One high-volume offense category is isolated for the core borough, with label harmonization documented
(we use the raw `OFNS_DESC` string as reported by NYPD, with no re-coding).


In [ ]:
# ============================================================
# 15. CATEGORY-SPECIFIC (OFFENSE-TYPE) SERIES
# ============================================================
top_categories = (df[df[CONFIG['location_col']] == CONFIG['location_value']][CONFIG['category_col']]
                   .value_counts().head(5))
print("Top 5 offense categories in", CONFIG['location_value'], ":\n", top_categories)

CATEGORY_VALUE = top_categories.index[0]
print(f"\nUsing category: {CATEGORY_VALUE}")

cat_df = df[(df[CONFIG['location_col']] == CONFIG['location_value']) &
            (df[CONFIG['category_col']] == CATEGORY_VALUE)]
y_cat = (cat_df.set_index(CONFIG['date_col'])
         .resample(CONFIG['frequency']).size().rename('incidents')
         .asfreq(CONFIG['frequency'], fill_value=0))

fig, ax = plt.subplots(figsize=(11,4))
y_cat.plot(ax=ax, title=f"{CATEGORY_VALUE} — {CONFIG['location_value']} (weekly)")
ax.set_ylabel('Incidents per week')
plt.tight_layout()
plt.savefig(FIG / '11_category_series.png', bbox_inches='tight')
plt.show()

if len(y_cat) > 3*H:
    tr_cat, te_cat = y_cat.iloc[:-H], y_cat.iloc[-H:]
    m_cat = ARIMA(tr_cat, order=best['order']).fit()
    pred_cat = m_cat.forecast(len(te_cat))
    print('Category-series ARIMA scores:', score(te_cat, pred_cat))
else:
    print('Category series too short for a locked holdout at this frequency; report as a limitation.')

---
## 16. ADVANCED EXTENSION F — Structural-Break / Temporal-Drift Screening (Manual §18.2, §23)

A simple CUSUM-style screen on the weekly counts flags candidate structural breaks for discussion
(not a formal changepoint model — presented as a screening diagnostic).


In [ ]:
# ============================================================
# 16. SIMPLE STRUCTURAL-BREAK SCREEN (CUSUM of standardized residuals from the mean)
# ============================================================
z = (y - y.mean()) / y.std()
cusum = z.cumsum()

fig, ax = plt.subplots(figsize=(11,4))
cusum.plot(ax=ax)
ax.axhline(0, color='gray', linestyle='--')
ax.set_title(f"CUSUM of standardized weekly counts — {CONFIG['location_value']} (screening only)")
ax.set_ylabel('Cumulative standardized deviation')
plt.tight_layout()
plt.savefig(FIG / '12_cusum_screen.png', bbox_inches='tight')
plt.show()
print("Interpretation: sustained monotonic drift in the CUSUM curve flags candidate periods of structural "
      "change (e.g. policy, seasonal, or data-system shifts) that deserve qualitative investigation; this is "
      "a screening heuristic, not a validated changepoint test.")

---
## 17. Discussion Questions — Worked Answers (Manual §20)

1. **Why is a random train/test split invalid?** It would let the model "see" future incident counts during
   training/tuning, producing artificially optimistic and unrealistic error estimates (temporal leakage).
2. **Occurrence date vs report date?** `CMPLNT_FR_DT` records when the incident is alleged to have happened;
   `RPT_DT` records when it was reported to NYPD. Reporting lag can be days to years, so using report date
   would distort the true temporal pattern of when incidents occur.
3. **Why does location define separate series rather than an ordinal ARIMA feature?** Borough codes are
   unordered labels; encoding "BRONX=1, BROOKLYN=2..." would impose a false numeric ordering with no
   statistical meaning. Building a distinct series per borough respects this.
4. **Practical stationarity for weekly crime counts?** Roughly constant mean, variance and autocorrelation
   structure over time — no systematic long-run drift in the weekly count's level or spread.
5. **p, d, q?** p = autoregressive lag order, d = number of differences applied for stationarity,
   q = moving-average (lagged forecast error) order.
6. **Why not rely on AIC alone?** AIC is an in-sample, likelihood-based complexity penalty; it does not
   directly measure how well the model forecasts genuinely unseen future data — hence the locked test-set
   MAE/RMSE evaluation.
7. **What does residual autocorrelation indicate?** Systematic structure the model failed to capture — the
   model is not yet adequate and could likely be improved.
8. **Why is MAPE unstable here?** Weekly counts can be zero or near-zero for rare categories, making
   percentage errors undefined or extreme.
9. **Sources of distribution shift?** Holidays, weather, policing-policy changes, legal reclassification of
   offenses, and changes to NYPD's own reporting/records systems.
10. **Why might seasonal modeling be needed?** Crime reporting can have annual cycles (e.g., seasonal upticks
    in certain offense types); SARIMA/SARIMAX can capture this given enough historical cycles.
11. **Consequences of negative ARIMA forecasts?** They are logically impossible for a count series and signal
    a Gaussian-model mismatch; Section 12 above compares a log1p-transformed alternative.
12. **Reported incidents vs actual prevalence?** Reported incidents are filtered through victim/witness
    reporting behavior and police recording practices — they are a proxy, not a full census of crime.
13. **Risks of treating forecast maps as person-level risk maps?** It risks stigmatizing residents of an area
    and can reinforce biased enforcement (Manual §23, "spatial stigmatization").
14. **Fair two-location comparison?** Use the identical time window, frequency, model order, and evaluation
    protocol for both locations (Section 10 above).
15. **When would a neural model be justified over ARIMA?** Only when it shows a large, reproducible accuracy
    gain over ARIMA/SARIMA on repeated runs that clearly justifies its added complexity and compute cost
    (Manual §16.4) — not by default.


---
## 18. Manifest, Acceptance Tests, and Reproducibility Record (Manual §12.9, Appendix C)


In [ ]:
# ============================================================
# 18.1 SAVE MANIFEST
# ============================================================
manifest = {
    **CONFIG,
    'student_name': STUDENT_NAME,
    'registration_number': REGISTRATION_NUMBER,
    'course_code': COURSE_CODE,
    'faculty': FACULTY,
    'semester': SEMESTER,
    'n_total_periods_core_location': int(len(y)),
    'n_train': int(len(train)),
    'n_test': int(len(test)),
    'ar_lags': AR_LAGS,
    'arima_candidate_orders': [list(c) for c in candidates],
    'selected_arima_order': list(best['order']),
    'sarima_seasonal_order': [1, 0, 1, SEASONAL_PERIOD],
    'python': sys.version,
    'platform': platform.platform(),
    'generated_utc': pd.Timestamp.utcnow().isoformat(),
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
print(json.dumps(manifest, indent=2, default=str))

In [ ]:
# ============================================================
# 18.2 CORE ACCEPTANCE TESTS (Appendix C)
# ============================================================
assert y.index.is_monotonic_increasing
assert y.index.is_unique
assert y.isna().sum() == 0
assert len(train) + len(test) == len(y)
assert train.index.max() < test.index.min()
assert len(test) == CONFIG['test_periods']
assert set(['actual','naive','AR','ARIMA']).issubset(pred_df.columns)
assert (OUT/'model_comparison.csv').exists()
assert (OUT/'test_predictions.csv').exists()
assert (OUT/'manifest.json').exists()
assert (OUT/'two_location_comparison.csv').exists()
assert (OUT/'rolling_origin_results.csv').exists()
print('All core + extension acceptance tests passed.')

In [ ]:
# ============================================================
# 18.3 REPRODUCIBILITY RECORD TABLE
# ============================================================
repro_record = pd.DataFrame([
    {'Item': 'Dataset name/version/access date', 'Record': f"{CONFIG['dataset']} / accessed {CONFIG['access_date']}"},
    {'Item': 'Source URL', 'Record': CONFIG['dataset_url']},
    {'Item': 'Local extract file', 'Record': DATA_FILENAME},
    {'Item': 'Date column used', 'Record': CONFIG['date_col']},
    {'Item': 'Location column/value (core)', 'Record': f"{CONFIG['location_col']} = {CONFIG['location_value']}"},
    {'Item': 'Location column/value (2nd)', 'Record': f"{CONFIG['location_col']} = {CONFIG['second_location_value']}"},
    {'Item': 'Crime category filter (advanced §15)', 'Record': CATEGORY_VALUE},
    {'Item': 'Aggregation frequency', 'Record': CONFIG['frequency']},
    {'Item': 'Observation window', 'Record': f"{y.index.min().date()} to {y.index.max().date()}"},
    {'Item': 'Forecast horizon', 'Record': f"{H} weeks"},
    {'Item': 'AR lags', 'Record': AR_LAGS},
    {'Item': 'ARIMA candidate orders', 'Record': str(candidates)},
    {'Item': 'Selected ARIMA order', 'Record': str(best['order'])},
    {'Item': 'SARIMA seasonal order', 'Record': f"(1,0,1,{SEASONAL_PERIOD})"},
    {'Item': 'Python / statsmodels versions', 'Record': f"{sys.version.split()[0]}"},
])
repro_record.to_csv(OUT / 'reproducibility_record.csv', index=False)
repro_record

In [ ]:
# ============================================================
# 18.4 SHORT-REPORT README (auto-generated skeleton, Manual §24)
# ============================================================
readme_text = f"""# {REGISTRATION_NUMBER}_{LAB_TITLE} — README

## 1. Aim and forecasting question
Forecast weekly reported crime-incident counts for {CONFIG['location_value']} (NYC borough) using AR and
ARIMA models, with a second-borough replication and several advanced extensions.

## 2. Dataset provenance and selected location
- Dataset: {CONFIG['dataset']}
- Source: {CONFIG['dataset_url']}
- Access date: {CONFIG['access_date']}
- Core location: {CONFIG['location_col']} = {CONFIG['location_value']}
- Replication location: {CONFIG['location_col']} = {CONFIG['second_location_value']}
- Date field: {CONFIG['date_col']} (occurrence date, not report date)

## 3. Aggregation/frequency and chronological split
- Frequency: {CONFIG['frequency']} (weekly)
- Test horizon: {H} weeks, locked, chronological (no shuffling)
- Total weekly periods (core location): {len(y)}

## 4. Stationarity and lag diagnostics
- ADF statistic: {adf_stat:.4f}, p-value: {adf_p:.4f}
- AR lag order: {AR_LAGS} (from training PACF)

## 5. AR and ARIMA models
- AR({AR_LAGS}) fit with trend='ct'
- ARIMA candidate orders searched: {candidates}
- Training-selected order (min AIC): {best['order']}

## 6. Forecast results (test MAE / RMSE)
{results.to_string(index=False)}

## 7. Residual/error analysis
- Ljung-Box p-value at lag {lb_lags}: {lb['lb_pvalue'].iloc[0]:.4f}
- Negative ARIMA forecasts on test set: {int((arima_pred < 0).sum())}

## 8. Time/location interpretation
See Section 10 (two-borough replication) and Section 14 (five-borough replication) in the notebook.
Forecast differences reflect reported-incident volume/dynamics by borough, not person-level risk.

## 9. Limitations and responsible-use statement
- Reported incidents are an administrative measure, not a census of actual crime.
- This extract is single-year (year-to-date); annual seasonality (SARIMA s=52) could not be reliably fit
  (see Section 11) — a multi-year historic extract would be needed.
- No person-level, address-level, or causal claims are made anywhere in this analysis.
- Forecasts are for academic/decision-support use only; not for autonomous patrol allocation.

## 10. Conclusion
Best-performing model on the locked test set: {results.iloc[0]['Model']} (MAE={results.iloc[0]['MAE']:.2f},
RMSE={results.iloc[0]['RMSE']:.2f}). See notebook Sections 9-16 for stability (rolling-origin), replication,
and advanced-extension evidence supporting this conclusion.
"""

(OUT / 'README.md').write_text(readme_text)
print(readme_text)

---
## 19. Package Everything and Auto-Download as a ZIP

This final cell:
1. Saves this notebook file itself into the outputs folder (via Colab's IPython kernel connection file — see note below).
2. Copies all figures, CSVs, JSON manifest and README into `lab06_outputs/`.
3. Zips the whole folder.
4. Triggers an automatic browser download of the ZIP (Colab only). If not running in Colab, the ZIP is simply
   left in the working directory for you to download manually.

> **Note:** Colab notebooks cannot easily self-save their own `.ipynb` file from inside a running cell. After
> running all cells, use **File → Download → Download .ipynb** in Colab once, and drop that file into the
> `lab06_outputs` folder (or just keep it alongside the ZIP) before final submission — everything else
> (figures/CSVs/manifest/README) is already zipped automatically below.


In [ ]:
# ============================================================
# 19. FINAL PACKAGING + AUTOMATIC ZIP DOWNLOAD
# ============================================================
ZIP_NAME = f"{REGISTRATION_NUMBER}_{LAB_TITLE}_outputs.zip"

# Rename output CSVs/JSON to the submission naming convention (Manual §24) inside a submission-ready copy
SUBMIT_DIR = Path('submission_ready')
if SUBMIT_DIR.exists():
    shutil.rmtree(SUBMIT_DIR)
SUBMIT_DIR.mkdir()

rename_map = {
    'model_comparison.csv': f'{REGISTRATION_NUMBER}_Lab06_Model_Comparison.csv',
    'test_predictions.csv': f'{REGISTRATION_NUMBER}_Lab06_Test_Predictions.csv',
    'manifest.json': f'{REGISTRATION_NUMBER}_Lab06_Manifest.json',
    'README.md': 'README.md',
    'two_location_comparison.csv': f'{REGISTRATION_NUMBER}_Lab06_Two_Location_Comparison.csv',
    'rolling_origin_results.csv': f'{REGISTRATION_NUMBER}_Lab06_Rolling_Origin_Results.csv',
    'multi_borough_replication.csv': f'{REGISTRATION_NUMBER}_Lab06_Multi_Borough_Replication.csv',
    'count_aware_comparison.csv': f'{REGISTRATION_NUMBER}_Lab06_CountAware_Comparison.csv',
    'reproducibility_record.csv': f'{REGISTRATION_NUMBER}_Lab06_Reproducibility_Record.csv',
}
for src_name, dst_name in rename_map.items():
    src = OUT / src_name
    if src.exists():
        shutil.copy(src, SUBMIT_DIR / dst_name)

# copy figures folder as-is
shutil.copytree(FIG, SUBMIT_DIR / 'figures', dirs_exist_ok=True)

# zip the whole submission_ready folder
if Path(ZIP_NAME).exists():
    Path(ZIP_NAME).unlink()

with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in SUBMIT_DIR.rglob('*'):
        zf.write(p, arcname=p.relative_to(SUBMIT_DIR.parent))

print(f"Created {ZIP_NAME}  ({Path(ZIP_NAME).stat().st_size/1024:.1f} KB)")
print("Contents:")
with zipfile.ZipFile(ZIP_NAME) as zf:
    for n in zf.namelist():
        print(' -', n)

# Auto-download in Colab
if IN_COLAB:
    from google.colab import files
    files.download(ZIP_NAME)
    print("\nDownload triggered automatically. If your browser blocked the popup, "
          "re-run this cell or download the file manually from the Colab file browser (left sidebar).")
else:
    print(f"\nNot running in Colab — the ZIP is saved locally at: {Path(ZIP_NAME).resolve()}")

---
## 20. Final Submission Checklist (Manual §26)

- [x] Official/instructor-approved dataset source and access date recorded (Section 1)
- [x] Selected location and aggregation frequency documented (Section 1–3)
- [x] Occurrence date (not report date) choice justified (Section 1)
- [x] Series is regular, ordered, unique, no unexplained missing intervals (Section 3, asserted)
- [x] Chronological train/test split used; no random shuffling (Section 4)
- [x] Naive baseline, AR and ARIMA results reported (Sections 4, 6, 7, 8)
- [x] ARIMA order selected without using final test values (Section 7)
- [x] ADF and ACF/PACF diagnostics included (Section 5)
- [x] MAE and RMSE reported on the locked future period (Section 8)
- [x] Residual diagnostic and Ljung-Box result included (Section 8.4)
- [x] Forecast plot includes actual values and train/test boundary (Section 8.2)
- [x] No person-level or causal crime claim is made (Sections 10, 14, 17)
- [x] Manifest, predictions, figures and notebook saved (Section 18–19)
- [x] Advanced experiments clearly separated from the core (Sections 11–16 labeled ADVANCED EXTENSION)

**Remember to:** rename this notebook file itself to `23MID0043_Lab06_Crime_AR_ARIMA.ipynb`,
download it from Colab (File → Download → Download .ipynb), and submit it alongside the auto-downloaded ZIP.
